# 00 — All market results for the seven trade files

The earlier pilot involved two markets, but the supplied settlement export already contains one row for **all 78,543 observed markets**. This notebook audits that complete export and separates recorded settlements from missing results. Its snapshot was extracted on **October 19, 2026, UTC**.

There are **13,794 recorded binary settlements** and **64,749 markets without a usable recorded result** in this snapshot. Missing records do not establish that the real-world market is unresolved; they remain unknown for accounting.

Run order for a complete rebuild: this notebook → `01_rf_data_preparation.ipynb` → `02_rf_models.ipynb` → `03_rf_validation.ipynb`. The package also includes executed model results.

In [1]:
from pathlib import Path
import os, sys, json
import pandas as pd
from IPython.display import display
here = Path.cwd().resolve()
folder = next((p for base in [here, *here.parents] for p in [base, base/'02_notebooks'] if (p/'prepare_full_model_inputs.py').is_file()), None)
if folder is None:
    raise FileNotFoundError('Start Jupyter inside the extracted wallet_full_analysis folder.')
sys.path.insert(0, str(folder))
PACKAGE_ROOT = folder.parent
from prepare_full_model_inputs import extract_all_results, build_inputs
SOURCE_METADATA = Path(os.environ.get('WALLET_MARKET_RESULTS', str(PACKAGE_ROOT/'data/weekly_market_metadata_complete.csv')))
print('Full result snapshot:', SOURCE_METADATA)

Full result snapshot: /workspace/scratch/2f27ad4cbed5/output/wallet_full_analysis/data/weekly_market_metadata_complete.csv


## Audit and export the complete matching market universe

The check compares market slugs with the original 78,543-market index. It retains results resolved after the trading week and keeps missing records. It also checks consistency among `resolved_yes`, the ordered outcome names, the supplied winner and the $1/$0 payout.

This verifies the database export's internal mapping. It does not independently repeat the earlier study's on-chain verification.

In [2]:
metadata, candidates = extract_all_results(SOURCE_METADATA, PACKAGE_ROOT)
coverage = json.loads((PACKAGE_ROOT/'04_results/market_results/coverage_summary.json').read_text())
display(pd.DataFrame([coverage]).T.rename(columns={0:'value'}))
display(metadata.loc[metadata.result_available_for_accounting, ['market','winning_outcome','resolved_at']].head(10))

Market coverage: {
  "markets_requested": 78543,
  "exported_unique_markets": 78543,
  "missing_market_rows": 0,
  "metadata_found": 78518,
  "metadata_missing": 25,
  "recorded_resolutions": 13794,
  "usable_binary_results": 13794,
  "no_usable_recorded_result": 64749,
  "original_candidate_markets": 2661,
  "original_candidate_trades": 239434,
  "candidate_wallet_market_pairs": 96184,
  "reproduces_original_candidate_counts": true,
  "candidate_outcome_name_mismatches": 0,
  "candidates_with_100plus_wallets": 265,
  "excluded_observed_trades_after_resolution": 28,
  "extracted_at_min": "2026-10-19 03:12:14.496408+00:00",
  "extracted_at_max": "2026-10-19 03:13:17.424113+00:00",
  "source_metadata_sha256": "e8db4bbd64b461563f5df24c55d9de46b773d93be2280b2bc65e0e81072210e8",
  "result_mapping_check": "Database resolved_yes, price_at_close, ordered outcomes and winner agree; not independent on-chain verification.",
  "retained_after_week_resolutions": true,
  "missing_results_recoded_as_

## Optional fresh database extraction using the same method

The scope is all slugs in `markets_in_yuzhe_files.csv`: **79 batches of up to 1,000 markets**. LEFT JOINs preserve missing metadata/results. No settlement-date filter is applied. This does not call the public trades API or download more raw trades.

Disabled by default. If you need a newer snapshot, install `requirements_database.txt`, enable this cell and enter the course PostgreSQL password in the hidden local prompt. The password is neither saved nor printed. The live database query was not executed during preparation of this artifact.

In [3]:
REFRESH_FROM_DATABASE = False
if REFRESH_FROM_DATABASE:
    from full_scope_query import extract_market_settlements
    fresh = extract_market_settlements(market_file=PACKAGE_ROOT/'data/markets_in_yuzhe_files.csv', project_dir=PACKAGE_ROOT, batch_size=1000)
    metadata, candidates = extract_all_results(fresh, PACKAGE_ROOT)
    print('Fresh full-scope export:', fresh)
else:
    print('Using the supplied full export; no new database connection was made.')

Using the supplied full export; no new database connection was made.


## Rebuild the original modeling inputs if needed

The original candidate rule recovers 2,661 markets: recorded binary settlements, scheduled start and recorded resolution within the supplied week, and no observed trade after recorded resolution. There is no minimum activity requirement and markets with negative-inventory wallets are retained at this stage. Those individual wallet labels become unknown later.

The reconstruction recovers 239,434 candidate trades and 96,184 wallet–market pairs, of which 89,736 have conditional gross-profit labels. It calculates first-batch entry features and strictly prior all-market history from all seven raw files.

The code preserves conservative event families and merges exact course-database event-slug links. Historical publication timestamps remain unverified. Refreshed metadata that changes the candidate universe is exported, but the original model rebuild stops until that cohort change is deliberately reviewed.

In [4]:
REBUILD_MODEL_INPUTS = False
# To rebuild, place all seven original trades-*.parquet files in data/weekly_trades,
# then set this flag True. The package includes cached derived inputs already.
if REBUILD_MODEL_INPUTS:
    build_inputs(SOURCE_METADATA, PACKAGE_ROOT)
else:
    print('Cached full-model inputs are included. Rebuild skipped.')

Cached full-model inputs are included. Rebuild skipped.


## Read the results

- `data/weekly_market_metadata_complete.csv`: complete 78,543-market metadata/results export.
- `04_results/market_results/recorded_market_results.csv`: 13,794 recorded settlements.
- `04_results/market_results/markets_without_recorded_results.csv`: missing result records.
- `04_results/market_results/all_market_result_coverage.csv`: coverage and original-candidate membership for every market.
- `data/candidate_markets_preliminary.csv`: the original 2,661-market universe.
- `04_results/random_forest/full_data_run/`: fitted comparisons, predictions, importance and plots.

The new prediction comparisons reuse the study's previously inspected test period and remain exploratory. Labels are conditional gross profitability, not verified net returns or insider status.